In [50]:
%pip install "vllm>=0.8.5" mcp ddgs smolagents markdownify

Note: you may need to restart the kernel to use updated packages.


In [51]:
!nvidia-smi

Mon Feb 23 18:03:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 581.29                 Driver Version: 581.29         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4070 ...  WDDM  |   00000000:01:00.0  On |                  N/A |
| 33%   46C    P2             29W /  220W |    9075MiB /  12282MiB |      1%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [52]:
import typing as tp
import subprocess

# Model setup

**Visit https://openrouter.ai/ to create an account, generate api key and search for models**

In [53]:
PROVIDER = 'OpenRouter' # OpenRouter or vLLM
# PROVIDER = 'vLLM' # OpenRouter or vLLM

In [54]:
from getpass import getpass
if PROVIDER == 'vLLM':
    KEY = getpass("Your openrouter api key: ")
    HOST = "127.0.0.1"
    PORT = "8999"
    MODEL = "Qwen/Qwen3-4B-Instruct-2507"
    URL = f"http://{HOST}:{PORT}/v1"
else:
    MODEL = "deepseek/deepseek-r1-0528:free"
    KEY = getpass("Your openrouter api key: ")
    URL = f"https://openrouter.ai/api/v1"


In [55]:
if PROVIDER == 'vLLM':
    vllm_server = subprocess.Popen([
        "python",
        "-m", "vllm.entrypoints.openai.api_server",
        "--model", MODEL,
        "--max_model_len", "16384",
        "--host", HOST,
        "--port", PORT,
        "--gpu_memory_utilization", "0.6",
        "--max_num_seqs", "16",
    ])
    #  python -m vllm.entrypoints.openai.api_server --model Qwen/Qwen3-4B-Instruct-2507 --max_model_len 16384 --host 127.0.0.1 --port 8999 --gpu_memory_utilization 0.6 --max_num_seqs 16 --temperature 0.9

# vllm_server.terminate() — остановить


In [ ]:
from openai import OpenAI

client = OpenAI(
  base_url=URL,
  api_key=KEY,
)

response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Hi"}],
)

print(response.choices[0].message)


ChatCompletionMessage(content='Hi there! 👋 How can I help you today?', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning='Hmm, the user just said "Hi" - that\'s a simple greeting. They might be testing if I\'m responsive or starting a new conversation. \n\nSince it\'s such a short message, I should keep my reply friendly and open-ended to encourage further interaction. A warm greeting with a question about how I can help would be appropriate here. \n\nI don\'t need to overcomplicate it - just acknowledge their greeting and pivot to offering assistance. The tone should be welcoming but professional.', reasoning_details=[{'format': 'unknown', 'index': 0, 'type': 'reasoning.text', 'text': 'Hmm, the user just said "Hi" - that\'s a simple greeting. They might be testing if I\'m responsive or starting a new conversation. \n\nSince it\'s such a short message, I should keep my reply friendly and open-ended to encourage further interact

# How to write tools? How to use MCP?

## Tools as functions

In [58]:
from ddgs import DDGS

def duckduckgo_search(query: str, max_results: int = 5):
    """
    Function for WEB searching using duckduckgo engine for provided query. Returns max_result_pages with snippets. 
    """
    with DDGS() as ddgs:
        results = ddgs.text(query, max_results=max_results, )
        return list(results)


In [ ]:
duckduckgo_search('yandex data school')

Impersonate 'chrome_119' does not exist, using 'random'


[{'title': 'Yandex School of Data Analysis',
  'href': 'https://grokipedia.com/page/Yandex_School_of_Data_Analysis',
  'body': 'Yandex School of Data Analysis (also known as ShAD or Школа анализа данных Яндекса) is an independent educational institution founded by Yandex in 2007 in Moscow, Russia. It provides advanced postgraduate programs focused on data analysis, machine learning, computer science, and related fields, known for its rigorous selection process and high-quality training in data science and AI. The school operates as a non-profit project aimed at preparing specialists for the tech industry.'},
 {'title': 'School of data analysis',
  'href': 'https://dataschool.yandex.com/',
  'body': 'A joint educational initiative of the Yandex School of Data Analysis, JetBrains and the Computer Science club. CS teaches software development, modern computer science, and data analysis.'},
 {'title': 'Yandex School of Data Analysis - GitHub',
  'href': 'https://github.com/yandexdataschool

In [59]:
import re
import requests
from markdownify import markdownify
from requests.exceptions import RequestException
def get_webpage_content(url: str) -> str:
    try:
        response = requests.get(url)
        response.raise_for_status()

        # Convert the HTML content to Markdown
        markdown_content = markdownify(response.text).strip()
        # Remove multiple line breaks
        markdown_content = re.sub(r"\n{3,}", "\n\n", markdown_content)

        return markdown_content

    except RequestException as e:
        return f"Error fetching the webpage: {str(e)}"
    except Exception as e:
        return f"An unexpected error occurred: {str(e)}"

In [60]:
# Implement function to get top-5 results with full text
def websearch_full_text(query, top_k=1) -> dict[str, tp.Any]:
    search_results = duckduckgo_search(query, top_k)
    for result in search_results:
        result['content'] = get_webpage_content(result["href"])
    return search_results

In [61]:
websearch_full_text("Agent systems", top_k=3)

Impersonate 'edge_127' does not exist, using 'random'


[{'title': 'Agere Systems',
  'href': 'https://en.wikipedia.org/wiki/Agere_Systems',
  'body': 'Agere Systems, Inc. was an integrated circuit components company based in Allentown, Pennsylvania. Spun out of Lucent Technologies in 2002, Agere was merged into LSI Corporation in 2007. LSI was in turn acquired by Avago Technologies in 2014. In early 2016, Avago acquired the former Broadcom Corporation, and took on the name Broadcom Inc.',
  'content': 'Error fetching the webpage: 403 Client Error: Forbidden for url: https://en.wikipedia.org/wiki/Agere_Systems'},
 {'title': 'Multi- agent system - Wikipedia',
  'href': 'https://en.wikipedia.org/wiki/Multi-agent_system',
  'body': 'Autonomy: agents are at least partially independent, self-aware, autonomous. Local views: no agent has a full global view, or the system is too complex for an agent to exploit such...',
  'content': 'Error fetching the webpage: 403 Client Error: Forbidden for url: https://en.wikipedia.org/wiki/Multi-agent_system'},

## Run MCP Server

## Connect MCP client

In [ ]:
import asyncio
from typing import Optional
from contextlib import AsyncExitStack

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

In [ ]:
class MCPClient:
    def __init__(self):
        self.session: Optional[ClientSession] = None
        self.exit_stack = AsyncExitStack()
        self.tools = []

    async def connect_to_server(self, server_script_path: str):
        command = "python"
        server_params = StdioServerParameters(
            command=command,
            args=[server_script_path],
            env=None
        )

        stdio_transport = await self.exit_stack.enter_async_context(stdio_client(server_params, errlog=None))
        self.stdio, self.write = stdio_transport
        self.session = await self.exit_stack.enter_async_context(ClientSession(self.stdio, self.write))
        await self.session.initialize()
        response = await self.session.list_tools()
        self.tools = response.tools

    def list_tools(self):
        print("\nConnected to server with tools:", [tool.name for tool in self.tools])
        return self.tools

    async def call_tool(self, tool_name, args):
        result = await self.session.call_tool(tool_name, args)
        return result



In [ ]:
client = MCPClient()
await client.connect_to_server('./ysda_tools.py')
tools = client.list_tools()


Connected to server with tools: ['add', 'subtract', 'multiply', 'divide', 'vector_add', 'vector_subtract', 'vector_dot', 'vector_elementwise_multiply', 'matrix_add', 'matrix_subtract', 'matrix_multiply', 'matrix_transpose']


In [ ]:
tools[0]

Tool(name='add', title=None, description='', inputSchema={'properties': {'a': {'title': 'A', 'type': 'number'}, 'b': {'title': 'B', 'type': 'number'}}, 'required': ['a', 'b'], 'title': 'addArguments', 'type': 'object'}, outputSchema={'properties': {'result': {'title': 'Result', 'type': 'number'}}, 'required': ['result'], 'title': 'addOutput', 'type': 'object'}, icons=None, annotations=None, meta=None, execution=None)

# Tool calls example

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_youtube_captions",
            "description": "Fetch YouTube captions for a given video ID",
            "parameters": {
                "type": "object",
                "properties": {
                    "video_id": {"type": "string"},
                    "lang": {"type": "string", "default": "en"}
                },
                "required": ["video_id"]
            }
        }
    }
]


In [ ]:
import inspect
import typing

PYTHON_TO_JSON = {
    str: "string",
    int: "integer",
    float: "number",
    bool: "boolean",
    list: "array",
    dict: "object",
}

def create_tool_description(func):
    sig = inspect.signature(func)
    doc = inspect.getdoc(func) or ""
    params_schema = {"type": "object", "properties": {}, "required": []}

    for name, param in sig.parameters.items():
        annotation = param.annotation
        if annotation in PYTHON_TO_JSON:
            json_type = PYTHON_TO_JSON[annotation]
        else:
            json_type = "string"  # fallback

        entry = {"type": json_type}

        if param.default is not inspect.Parameter.empty:
            entry["default"] = param.default
        else:
            params_schema["required"].append(name)

        params_schema["properties"][name] = entry

    return {
        "type": "function",
        "function": {
            "name": func.__name__,
            "description": doc,
            "parameters": params_schema,
        },
    }


In [ ]:
create_tool_description(duckduckgo_search)

{'type': 'function',
 'function': {'name': 'duckduckgo_search',
  'description': 'Function for WEB searching using duckduckgo engine for provided query. Returns max_result_pages with snippets. ',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string'},
    'max_results': {'type': 'integer', 'default': 5}},
   'required': ['query']}}}

In [64]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "Always talk in shakespearean english, always follow user's instructions."},
        {"role": "user", "content": "Hi"},
        {"role": "user", "content": "When Queen anne's revenge sunk"}],
    tools = [create_tool_description(duckduckgo_search)]
)

print(response.choices[0].message)

ChatCompletionMessage(content="I shall search for knowledge regarding the sinking of that infamous vessel, Queen Anne's Revenge.", refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_16aae7fac3d14071a97be77d', function=Function(arguments='{"query": "When did Queen Anne\'s Revenge sink Blackbeard ship", "max_results": 10}', name='duckduckgo_search'), type='function', index=0)], reasoning='The user asked "When Queen anne\'s revenge sunk". I need to interpret this as "When did Queen Anne\'s Revenge sink?" I should search for information about the sinking of Blackbeard\'s ship Queen Anne\'s Revenge. I\'ll use the duckduckgo_search function to find accurate historical information.', reasoning_details=[{'format': 'unknown', 'index': 0, 'type': 'reasoning.text', 'text': 'The user asked "When Queen anne\'s revenge sunk". I need to interpret this as "When did Queen Anne\'s Revenge sink?" I should search for

# ReAct Agent from scratch

In [65]:
instruction = """Solve a question answering task with interleaving Thought, Action, Observation steps. Thought can reason about the current situation, and Action can be three types:
(1) Search[query], which searches the web for the query and returns url, title and small snippet for 5 relevant pages.
(2) Visit web-page[link], which returns content of the page provided.
(3) Finish[answer], which returns the answer and finishes the task.
"""


final_prompt = """
Question: What is the capital of France?
Thought: I need to find the capital city of France.
Action: Search[capital of France]
Observations: 
1. https://en.wikipedia.org/wiki/Paris - Paris is the capital and most populous city of France
2. https://www.britannica.com/place/Paris - Paris, city and capital of France
3. https://www.thoughtco.com/paris-capital-city-france-1434323 - Paris has been the capital of France since 987 AD
4. https://www.visitparis.com/en/ - Paris, the capital of France, welcomes you
5. https://www.worldatlas.com/articles/what-is-the-capital-of-france.html - The capital of France is Paris
Thought: The search results consistently indicate Paris is the capital. I can answer confidently.
Action: Finish[Paris]

Question: Who wrote the novel "1984"?
Thought: I need to find the author of the novel "1984".
Action: Search[author of 1984 novel]
Observations: 
1. https://en.wikipedia.org/wiki/George_Orwell - George Orwell wrote 1984, published in 1949
2. https://www.britannica.com/biography/George-Orwell - George Orwell's works include Animal Farm and 1984
3. https://www.george-orwell.org/ - George Orwell (1903-1950) was the author of 1984
4. https://www.nytimes.com/topic/person/george-orwell - George Orwell, author of 1984 and Animal Farm
5. https://www.theguardian.com/books/georgeorwell - George Orwell, whose real name was Eric Blair, wrote 1984
Thought: Multiple sources confirm George Orwell wrote 1984.
Action: Finish[George Orwell]

Question: What is the population of Tokyo?
Thought: I need to find the current population of Tokyo, Japan.
Action: Search[Tokyo population 2024]
Observations: 
1. https://www.metro.tokyo.lg.jp/english/ - Tokyo's population is approximately 14 million within the 23 special wards
2. https://worldpopulationreview.com/cities/japan/tokyo - Tokyo population: 37.4 million (metropolitan area) as of 2024
3. https://www.statista.com/topics/6341/tokyo/ - Greater Tokyo Area has 37.3 million inhabitants
4. https://www.britannica.com/place/Tokyo - Tokyo's prefecture population is about 14 million, metropolitan area over 37 million
5. https://www.toukei.metro.tokyo.lg.jp/english/ - Tokyo Metropolis population estimate: 14,094,034 as of January 2024
Thought: There's a distinction between the city proper (Tokyo Metropolis) and the Greater Tokyo Area. The question likely means the city proper.
Action: Finish[14 million (Tokyo Metropolis proper)]

Question: When was the first iPhone released?
Thought: I need to find the release date of the first iPhone.
Action: Search[first iPhone release date]
Observations: 
1. https://www.apple.com/newsroom/2007/01/09Apple-Reinvents-the-Phone-with-iPhone/ - Apple introduced iPhone on January 9, 2007
2. https://en.wikipedia.org/wiki/IPhone_(1st_generation) - The first iPhone was released on June 29, 2007 in the US
3. https://www.macworld.com/article/670849/history-of-iphone.html - iPhone announced January 2007, released June 29, 2007
4. https://www.cnet.com/tech/mobile/original-iphone-release-date-2007/ - Original iPhone went on sale June 29, 2007
5. https://www.theverge.com/2017/6/29/15890462/iphone-10-year-anniversary-retrospective - iPhone launched June 29, 2007
Thought: The announcement was January 2007, but the actual release to consumers was June 29, 2007. The question asks when it was released.
Action: Finish[June 29, 2007]

Question: What is the chemical symbol for gold?
Thought: I need to find the chemical element symbol for gold.
Action: Search[chemical symbol for gold]
Observations: 
1. https://pubchem.ncbi.nlm.nih.gov/element/Gold - Gold (Au) atomic number 79
2. https://www.britannica.com/science/gold-chemical-element - Gold (Au) from Latin aurum
3. https://www.rsc.org/periodic-table/element/79/gold - Gold symbol Au, from Latin aurum meaning shining dawn
4. https://en.wikipedia.org/wiki/Gold - Gold is a chemical element with the symbol Au
5. https://www.chemicool.com/elements/gold.html - Element Gold (Au), Group 11, Atomic Number 79
Thought: All sources confirm gold's chemical symbol is Au.
Action: Finish[Au]

Now, solve the following question:

Question: {When the siege of Tortuga happened?}
"""

In [66]:
def llm(prompt, stop=["\n"]):
    response = client.chat.completions.create(
      model=MODEL,
      prompt=prompt,
      temperature=0,
      max_tokens=100,
      top_p=1,
      frequency_penalty=0.0,
      presence_penalty=0.0,
      stop=stop
    )
    return response["choices"][0]["text"]

In [ ]:
def execute_action(action: str) -> str:
    pass


def call_react(question, prompt=final_prompt, to_print=True):
    prompt += question + "\n"
    n_calls, n_badcalls = 0, 0
    for i in range(1, 8):
        n_calls += 1
        # Generate thought
        thought_action = <YOUR CODE HERE>
        try:
            # Extract generated action
            thought, action = thought_action.strip().split(f"\nAction {i}: ")
        except Exception as e:
            # Error handling
        # Execute action
        ... = execute_action()
        # Create observation
        obs = obs.replace('\\n', '')
        step_str = f"Thought {i}: {thought}\nAction {i}: {action}\nObservation {i}: {obs}\n"
        # Add observation to history
    return r, info

In [ ]:
client.completions

# Multi-agent systems with smolagents

In [40]:
from smolagents import tool


@tool
def visit_webpage(link: str) -> str:
  """
  Visit web-page[link], which returns content of the page provided.
  Args:
    link: link to the webpage
  Returns:
    content of the webpage
  """
  return get_webpage_content(link)

In [41]:
from smolagents import (
    CodeAgent,
    OpenAIModel,
    ToolCallingAgent,
    WebSearchTool,
)


model = OpenAIModel(
    model_id=MODEL,
    api_base=URL,
    api_key=KEY,
)


web_agent = CodeAgent(
    tools=[WebSearchTool(), visit_webpage],
    model=model,
    max_steps=5,
    name="web_search_agent",
    description="Runs web searches for you.",
)

In [42]:
!nvidia-smi

Mon Feb 23 17:35:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 581.29                 Driver Version: 581.29         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4070 ...  WDDM  |   00000000:01:00.0  On |                  N/A |
|  0%   61C    P2             34W /  220W |    9115MiB /  12282MiB |      2%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [43]:
manager_agent = CodeAgent(
    tools=[],
    model=model,
    managed_agents=[web_agent],
)

In [44]:
answer = manager_agent.run(
    "How does ReAct agent works? What metrics were reported by authors?"
)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ How does ReAct agent works? What metrics were reported by authors?                                              │
│                                                                                                                 │
╰─ OpenAIModel - stepfun/step-3.5-flash:free ─────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Error in code parsing:
Your code snippet is invalid, because the regex pattern <code>(.*?)</code> was not found in it.
            Here is your code snippet:
             query.
</think>
I'll help you understand how ReAct agent works and what metrics were reported by the authors. Let me start by 
searching for information about the ReAct framework.Thought: I need to search for information about the ReAct 
agent. I'll use web_search to find the original paper and its details.<tool_call>
<function=web_search_agent>
<parameter=task>
Find information about ReAct agent: "ReAct: Synergizing Reasoning and Acting in Language Models" paper. Explain how
it works and what metrics were reported by the authors.
</parameter>
<parameter=additional_args>
{}
</parameter>
</function>
</tool_call></code>
            Make sure to include code with the correct pattern, for instance:
            Thoughts: Your thoughts
            <code>
            # Your python code here
            </code>
Make sure to provide correct code blobs.

[Step 1: Duration 6.48 seconds| Input tokens: 2,190 | Output tokens: 308]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

...

[Step 2: Duration 70.91 seconds| Input tokens: 4,768 | Output tokens: 534]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 3 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer({                                                                                                   
      "how_react_works": "ReAct (Reasoning + Acting) is an approach that interleaves verbal reasoning traces with  
  task-specific actions in a unified framework. The model generates both reasoning steps (for planning, tracking,  
  and handling exceptions) and actions (to interact with external environments like Wikipedia, simulators, or      
  APIs) in an alternating manner. For reasoning-heavy tasks (QA), it uses explicit alternation between reasoning   
  and actions. For action-heavy tasks (embodied/robotics), it uses sparse reasoning where the model decides when   
  to reason. This synergy reduces hallucination, improves interpretability, and enables human collaboration.",     
                                                                                                                   
      "metrics_reported": {                                                                                        
          "hotpotqa": {"metric": "Exact Match (EM)", "react": 27.4, "react_cot_best": 35.1, "standard": 28.7,      
  "reason_only_cot": 29.4, "act_only": 25.7},                                                                      
          "fever": {"metric": "Accuracy", "react": 60.9, "react_cot_best": 64.6, "standard": 57.1,                 
  "reason_only_cot": 56.3, "act_only": 58.9},                                                                      
          "alfworld": {"metric": "Task Success Rate", "react": 71, "act_only": 45, "improvement": "+34%"},         
          "webshop": {"metric": "Task Success Rate", "react": 40, "act_only": 30.1, "improvement": "+10%"}         
      },                                                                                                           
                                                                                                                   
      "key_findings": [                                                                                            
          "ReAct outperformed reasoning-only (CoT), acting-only, and standard prompting baselines",                
          "Combining ReAct with Chain-of-Thought yielded best results on QA tasks (35.1 EM on HotpotQA, 64.6       
  accuracy on FEVER)",                                                                                             
          "Significant improvements in interactive decision-making: +34% on ALFWorld and +10% on WebShop over      
  act-only baselines",                                                                                             
          "ReAct showed strong data efficiency, using only 1-2 shot examples compared to imitation learning        
  requiring ~100k training samples",                                                                               
          "Fine-tuning smaller models (8B/62B) with ReAct trajectories showed transferability of the pattern"      
      ]                                                                                                            
  })                                                                                                               
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: {'how_react_works': 'ReAct (Reasoning + Acting) is an approach that interleaves verbal reasoning 
traces with task-specific actions in a unified framework. The model generates both reasoning steps (for planning, 
tracking, and handling exceptions) and actions (to interact with external environments like Wikipedia, simulators, 
or APIs) in an alternating manner. For reasoning-heavy tasks (QA), it uses explicit alternation between reasoning 
and actions. For action-heavy tasks (embodied/robotics), it uses sparse reasoning where the model decides when to 
reason. This synergy reduces hallucination, improves interpretability, and enables human collaboration.', 
'metrics_reported': {'hotpotqa': {'metric': 'Exact Match (EM)', 'react': 27.4, 'react_cot_best': 35.1, 'standard': 
28.7, 'reason_only_cot': 29.4, 'act_only': 25.7}, 'fever': {'metric': 'Accuracy', 'react': 60.9, 'react_cot_best': 
64.6, 'standard': 57.1, 'reason_only_cot': 56.3, 'act_only': 58.9}, 'alfworld': {'metric': 'Task Success Rate', 
'react': 71, 'act_only': 45, 'improvement': '+34%'}, 'webshop': {'metric': 'Task Success Rate', 'react': 40, 
'act_only': 30.1, 'improvement': '+10%'}}, 'key_findings': ['ReAct outperformed reasoning-only (CoT), acting-only, 
and standard prompting baselines', 'Combining ReAct with Chain-of-Thought yielded best results on QA tasks (35.1 EM
on HotpotQA, 64.6 accuracy on FEVER)', 'Significant improvements in interactive decision-making: +34% on ALFWorld 
and +10% on WebShop over act-only baselines', 'ReAct showed strong data efficiency, using only 1-2 shot examples 
compared to imitation learning requiring ~100k training samples', 'Fine-tuning smaller models (8B/62B) with ReAct 
trajectories showed transferability of the pattern']}

[Step 3: Duration 8.80 seconds| Input tokens: 9,003 | Output tokens: 1,192]

In [45]:
answer

{'how_react_works': 'ReAct (Reasoning + Acting) is an approach that interleaves verbal reasoning traces with task-specific actions in a unified framework. The model generates both reasoning steps (for planning, tracking, and handling exceptions) and actions (to interact with external environments like Wikipedia, simulators, or APIs) in an alternating manner. For reasoning-heavy tasks (QA), it uses explicit alternation between reasoning and actions. For action-heavy tasks (embodied/robotics), it uses sparse reasoning where the model decides when to reason. This synergy reduces hallucination, improves interpretability, and enables human collaboration.',
 'metrics_reported': {'hotpotqa': {'metric': 'Exact Match (EM)',
   'react': 27.4,
   'react_cot_best': 35.1,
   'standard': 28.7,
   'reason_only_cot': 29.4,
   'act_only': 25.7},
  'fever': {'metric': 'Accuracy',
   'react': 60.9,
   'react_cot_best': 64.6,
   'standard': 57.1,
   'reason_only_cot': 56.3,
   'act_only': 58.9},
  'alfwor

In [48]:
from IPython.display import display, Markdown, Latex
display(Markdown(answer["how_react_works"]))

ReAct (Reasoning + Acting) is an approach that interleaves verbal reasoning traces with task-specific actions in a unified framework. The model generates both reasoning steps (for planning, tracking, and handling exceptions) and actions (to interact with external environments like Wikipedia, simulators, or APIs) in an alternating manner. For reasoning-heavy tasks (QA), it uses explicit alternation between reasoning and actions. For action-heavy tasks (embodied/robotics), it uses sparse reasoning where the model decides when to reason. This synergy reduces hallucination, improves interpretability, and enables human collaboration.